In [2]:
from collections import Counter, deque
import re
import json
from functools import lru_cache

In [3]:
class BPETokenizerSimple:

    def __init__(self):
        # maps token_id to token_str (eg., {1124 : "some"})
        self.vocab = {}

        # maps token_str to token_id (eg., ({"some" : 1124}))
        self.inverse_vocab = {}

        # dictionary of BPE merges (eg. , {(token_id1, token_id2) : merged_token_id})
        self.bpe_merges = {}

        # dictionary like gpt-2
        self.bpe_rank = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from the sratch. 

        Args: 
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (str): A set of special tokens to include. 
        """

        # Pre tokenizing training text 
        tokens = self.pretokenize_text(text)

        # lets initialize our vocab with unique characters, including "Ġ" if present 
        # lets start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            ch for ch in sorted({char for token in tokens for char in token})
            if ch not in unique_chars
        )

        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i : char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char : i for i, char in enumerate(unique_chars)}

        # add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in unique_chars:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id
                
        # Tokenize each pre token into a token_id
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE steps 1-3: repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merges
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_tokens = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_tokens
            self.inverse_vocab[merged_tokens] = new_id


    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json is {token_str: id}; we want id->str and str->id
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
    
        # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # Must have <|endoftext|> at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
    
        # Provide a convenience alias for '\n' -> 198
        # Keep printable character 'Ċ' in vocab so BPE merges keep working
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Load GPT-2 merges and store ranks
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
    
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Safe to skip pairs whose symbols are not in vocab
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token Ids, with tiktoken style handling of special tokens

        Args: 
            text (str): the input text to encode.
            allowed_special (set or None): specical tokens to allow in the text. if None, special handling is disabled
        """

        specials_in_vocab = [
            tok
            for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
            ]

        if allowed_special is None:
            # nothing is allowed
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special token encountered in text {disallowed}")
        else:
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special token encountered in text {disallowed}")
        
        token_ids = []
        # If some specials are allowed, split around them and passthrough those ids
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)) + ")"

            last_index = 0
            for match in re.finditer(special_pattern, text):

                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))

                special_token = match.group(0)

                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"special token {special_token} not found in vocabulary")
                last_index = match.end()

            text = text[last_index:]

            # extra guard for any special literal left
            dissallowed = [
                tok
                for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
                ]

            if dissallowed:
                raise ValueError(f"Dissallowed special token {dissallowed} found in text")
                
        tokens = self.pretokenize_text(text)

        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
        
        return token_ids


    def decode(self, token_ids):
        """
        Decode a list of token ids back into a string.

        Args:
            token_ids (List[int]): the list of token ids to decode

        Returns:
            str: the decoded string
        """

        out = []
        for id in token_ids:
            if id not in self.vocab:
                raise ValueError(f"token id {id} not found in vocab")
            
            tok = self.vocab[id]
            if id == 198 or tok == "\n":
                out.append("\n")
            elif id == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        
        return "".join(out)


    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        save the vocabulary and the merges to the json files.

        Args: 
            vocab_path (str): path to save the vocabulary 
            bpe_merges_path (str): path to save the merges
        """

        # save vocabulary
        with open(vocab_path, "w", encoding='utf=8') as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # save bpe merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id} for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)


    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        load the vocabulary and the BPE merges from the json file.

        Args: 
            vocab_path (str): path to load the vocabulary
            bpe_merges_path (str): path to load the meerges
        """

        # load vocabulary 
        with open(vocab_path, "r", encoding="utf-8") as f:
                loaded_vocab = json.load(f)
                self.vocab = {int(id) : tok for id, tok in loaded_vocab.items()}
                self.inverse_vocab = {tok : int(id) for id, tok in loaded_vocab.items()}

        # load merges
        with open(bpe_merges_path, "r", encoding="utf-8") as f:
            loaded_merges = json.load(f)
            for merge in loaded_merges:
                pair = tuple(merge['pair'])
                new_id = merge['new_id']
                self.bpe_merges[pair] = new_id


    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)


    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE

        Args: 
            token (str): The token to tokenize 

        Returns:
            List (int): list of token ids after applying bpe 
        """
        # tokenize the token into characters
        token_ids = [self.inverse_vocab.get(char, None) for char in token]

        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"characters not found in vocab {missing_chars}")

        # if gpt2's merge is not loaded
        if not self.bpe_rank:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                i = 0
                new_tokens = []
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i+1])
                    if pair in self.bpe_merges:
                        new_tokens.append(self.bpe_merges[pair])
                        i+=2
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i+=1

                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                
                token_ids = new_tokens

            return token_ids
                
        # if gpt2's merge is loaded
        # first convert the token ids back to their string reprsentations
        symbols = [self.vocab[tok_id] for tok_id in token_ids]

        # repeatedly merge all the pairs of the lowest rank
        while True:
            # collect all adjacent pairs
            pairs = set(zip(symbols, symbols[1:]))

            if not pairs:
                break

            min_rank = float('inf')
            bigram = None

            for p in pairs:
                r = self.bpe_rank.get(p, float('inf'))
                if r < min_rank:
                    bigram = p
                    min_rank = r

            if bigram is None or bigram not in self.bpe_rank:
                break

            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols) - 1:
                # if we see the bigram at i then we will merge them 
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)
                    i+=2
                else:
                    new_symbols.append(symbols[i])
                    i+=1
            symbols = new_symbols

            if len(symbols) == 1:
                break
        
        merged_ids = [self.inverse_vocab[tok] for tok in symbols]
        return merged_ids
            



    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequence = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replace = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replace.append(new_id)
                    # remove the 2nd token of the pair, 1st one was already removed
                    dq.popleft()
                else:
                    replace.append(current)
            replaced_sequence.append(replace)
        return replaced_sequence



    @staticmethod 
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token in token_id_sequences
            for pair in zip(token, token[1:]) 
        )

        if not pairs:
            return None
        
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.") 
        

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == '\r':
                tokens.append('\r')
                continue
            if part == '\r\n':
                tokens.append('\r')
                continue
            if part == '\n':
                tokens.append('\n')  
                continue


            # Normal chunk with line breaks:
            # - If spaces are there before a word then add 'Ġ' as prefix to the first word
            #   and add rest of the spaces as single 'Ġ' 
            # - If spaces are after the word add 'Ġ' alone

            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))             # why +=, why not only = 
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append('Ġ')
                        tokens.append('Ġ' + word)
                        pending_spaces = 0
                    else:
                        tokens.append(word)

                for _ in range(pending_spaces):
                    tokens.append('Ġ')
        return tokens

In [4]:
import os
import requests


def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path
        
    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"failed to download {filename}. Error: {e}")
    return target_path 



verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=[".."]
)

with open(verdict_path, "r", encoding="utf-8") as f:
    text = f.read()



the-verdict.txt already exists in ..\the-verdict.txt


In [5]:
text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [8]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=50000, allowed_special={"<|endoftext|>"})

In [23]:
print(len(tokenizer.vocab))

1000


In [27]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[74, 361, 256, 310, 109, 98, 420, 397, 100, 256, 300, 428, 116, 121, 256, 519, 256, 699, 256, 299, 256, 808, 534]


In [30]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

[74, 361, 256, 310, 109, 98, 420, 397, 100, 256, 300, 428, 116, 121, 256, 519, 256, 699, 256, 299, 256, 808, 534, 257, 256]


In [31]:
print(tokenizer.decode(token_ids))

Jack  embraced  beauty  through  art  and  life.<|endoftext|> 


In [29]:
print(1+2)

3


In [33]:
for token in token_ids:
    print(f"{token} -> {tokenizer.decode([token])}")

74 -> J
361 -> ack
256 ->  
310 ->  e
109 -> m
98 -> b
420 -> ra
397 -> ce
100 -> d
256 ->  
300 ->  be
428 -> au
116 -> t
121 -> y
256 ->  
519 ->  through
256 ->  
699 ->  art
256 ->  
299 ->  and
256 ->  
808 ->  lif
534 -> e.
257 -> <|endoftext|>
256 ->  


In [34]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This  is  some  text  with \n  newline  characters.'

In [9]:
# save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path='vocab.json', bpe_merges_path='bpe_merges.txt')

In [36]:
#load tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path='vocab.json', bpe_merges_path='bpe_merges.txt')

3.3 Loading the original GPT-2 BPE tokenizer from OpenAI

In [5]:
search_directories = ["."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Ensure directories exist and download files if needed

paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

downloaded vocab.bpe to .\vocab.bpe
downloaded encoder.json to .\encoder.json


In [6]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"])

In [7]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, world. Is this-- a test?"

In [9]:
tk_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
tk_ids

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [10]:
tokenizer.decode(tk_ids)

'Hello, world. Is this-- a test?'